# Sequa: End-to-End LLM Interaction Recorder & Replayer

Welcome to the Sequa demonstration notebook! This notebook provides an interactive walkthrough to test and explore the features of **Sequa**, a tool for recording and replaying LLM responses (similar to VCR.py but optimized for LLMs).

## 1. Setup & Environment Verification

First, we append `src/` to Python's system path so that the local `sequa` package is importable, then load environment variables and verify imports.

In [1]:
import os
import sys

# Add the src directory to sys.path so that local 'sequa' is importable
sys.path.insert(0, os.path.abspath("src"))

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from sequa.cassette import cassette

# Load environment variables
load_dotenv()

# Ensure GROQ_API_KEY is available
if not os.getenv("GROQ_API_KEY"):
    print("Warning: GROQ_API_KEY is not set in environment!")
else:
    print("GROQ_API_KEY is successfully loaded.")

GROQ_API_KEY is successfully loaded.


### Optional Mock Mode Toggle

Since the `GROQ_API_KEY` in the workspace environment may be invalid or expired, we default `MOCK_MODE = True`. This mocks `ChatGroq.invoke` to return realistic mocked messages, allowing you to test Sequa's full context-manager, patching, normalizer, and CLI flows without calling the live API.

Set `MOCK_MODE = False` if you have a valid Groq API key and want to test with the live service.

In [2]:
# Mock/Real Mode Toggle
MOCK_MODE = False

from unittest.mock import MagicMock
from langchain_core.messages import AIMessage

if MOCK_MODE:
    print("Mock mode is enabled. ChatGroq.invoke will be mocked for demo/testing purposes.")
    if not hasattr(ChatGroq, "_original_invoke"):
        ChatGroq._original_invoke = ChatGroq.invoke
    
    def mock_invoke(self, input_data, *args, **kwargs):
        prompt = str(input_data)
        if "France" in prompt:
            content = "The capital of France is Paris."
        elif "2 + 2" in prompt:
            content = "2 + 2 is equal to 4."
        else:
            content = "Hello from mock live Groq!"
            
        return AIMessage(
            content=content,
            response_metadata={"model_name": self.model_name or "llama-3.1-8b-instant", "total_time": 0.5},
            usage_metadata={"input_tokens": 10, "output_tokens": 5, "total_tokens": 15},
            id="msg-mocked",
        )
    
    ChatGroq.invoke = mock_invoke
else:
    print("Real API mode is enabled. Using live Groq API.")
    if hasattr(ChatGroq, "_original_invoke"):
        ChatGroq.invoke = ChatGroq._original_invoke

Real API mode is enabled. Using live Groq API.


## 2. Recording LLM Interactions

We use Sequa as a context manager. By running code within the `with cassette(path="demo_cassettes/hello_run", mode="record"):` block, any call to `ChatGroq.invoke` will make a live (or mocked) call, capture the raw and canonical request/response, and write it to `demo_cassettes/hello_run.json`.

In [3]:
# Initialize ChatGroq model
model = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0.1)

# 1. Record Mode
print("Recording live call...")
with cassette(path="demo_cassettes/hello_run", mode="record"):
    res = model.invoke("Say 'Hello from Sequa!' in a short and friendly way.")
    print("\nResponse output:")
    print(res.content)

Recording live call...

Response output:
Hello from Sequa!


## 3. Replaying the Recorded Interaction

Now we can run the exact same call within `with cassette(path="demo_cassettes/hello_run", mode="replay"):`. Sequa matches the request inputs and configuration, intercepts the call, and instantly returns the cached response with zero latency or API cost.

In [4]:
# 2. Replay Mode
print("Replaying recorded call...")
with cassette(path="demo_cassettes/hello_run", mode="replay"):
    res_replayed = model.invoke("Say 'Hello from Sequa!' in a short and friendly way.")
    print("\nReplayed response output:")
    print(res_replayed.content)
    print("\nMetadata (model name):", res_replayed.response_metadata.get("model_name"))

Replaying recorded call...

Replayed response output:
Hello from Sequa!

Metadata (model name): openai/gpt-oss-120b


## 4. Automatic Patching (Dictionary Response Mode)

We can globally patch `ChatGroq` using `patch_langchain()`. When called outside of a context manager, the patched model returns a dictionary wrapping canonical request, canonical response, and raw output object.

In [5]:
from sequa.llm.adapters.patch_groq import patch_langchain

# Patch ChatGroq globally
patch_langchain()

# Call invoke outside of a cassette context - returns request/response dictionary wrapper
dict_result = model.invoke("What is 2 + 2?")
print("Intercepted result dictionary keys:", list(dict_result.keys()))
print("\nCanonical Request model:", dict_result["request"].model)
print("Canonical Response output:", dict_result["response"].output)

Intercepted result dictionary keys: ['request', 'response', 'raw']

Canonical Request model: openai/gpt-oss-120b
Canonical Response output: 2 + 2 = 4.


## 5. Advanced Features: Ignoring Fields

By default, any changes to request parameters (like `temperature`) will prevent a cache match. To allow matching despite changes in dynamic settings, you can pass `ignore_fields` to `cassette`.

In [6]:
# Record a response with temperature=0.1, setting ignore_fields=["temperature"]
with cassette("demo_cassettes/temp_flow", mode="record", ignore_fields=["temperature"]):
    model_low = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.1)
    model_low.invoke("State the capital of France.")

# Replay with temperature=0.9. It matches successfully because "temperature" is ignored!
with cassette("demo_cassettes/temp_flow", mode="replay", ignore_fields=["temperature"]):
    model_high = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.9)
    res_temp = model_high.invoke("State the capital of France.")
    print("Replayed successfully:", res_temp.content)

Replayed successfully: The capital of France is Paris.


## 6. Managing Cassettes with Sequa CLI

Sequa features a command-line interface to inspect, summarize, and clean recorded cassettes. Let's use the CLI directly from the notebook.

In [7]:
# Check stats: displays count, file size, and total saved API latency
!uv run sequa stats --path demo_cassettes

 Sequa Statistics
Total Cassettes:      2
Total Size on Disk:   3.95 KB (4049 bytes)
Total Latency Saved:  0.77 seconds (766.8 ms)


In [8]:
# Inspect: lists all saved cassettes, showing providers, models, and timestamps
!uv run sequa inspect --path demo_cassettes

Filename / Hash                          | Provider        | Model                     | Created At               
---------------------------------------------------------------------------------------------------------------
fbb1075ea85891758183d21b94e7381f5bf...   | langchain_groq  | llama-3.1-8b-instant      | 2026-07-11T15:55:27.580619+00:00
00d9037312e5e0e6c495fb4f250f7a23053...   | langchain_groq  | openai/gpt-oss-120b       | 2026-07-11T15:54:51.656308+00:00


### Redacting dynamic/volatile data for Git commits

When committing cassettes to Git, you don't want noise from changing latencies or timestamps. Sequa CLI's `clean` command handles this by removing latency fields and redacting timestamps.

In [9]:
# Run clean to sanitize latency and timestamps
!uv run sequa clean --path demo_cassettes --remove-latency --remove-timestamps

Successfully formatted/cleaned 2 cassettes.


In [10]:
# Check stats again (latency saved should now reflect 0.0 seconds as they have been redacted)
!uv run sequa stats --path demo_cassettes

 Sequa Statistics
Total Cassettes:      2
Total Size on Disk:   3.73 KB (3817 bytes)
Total Latency Saved:  0.00 seconds (0.0 ms)
